# HackatEOn

## Agenda

**Classification**

1. Meet the data
    1. Biromaru, Indonesia
    2. Grande-Terre, Guadeloupe
    3. Segmentation Masks
2. Classification task
    1. Random Forest Classifier
    2. Support Vector Machine
    3. Artificial Neural Network
    4. Data Preprocessing
    5. Spectral Information

**Segmentation**

3. Segmentation task
    1. Random Forest
    2. Fully Convolutional Network (FCN)
    3. U-Net
    4. Pre-trained Models
    5. Transfer Learning: Fine Tuning
    6. Transfer Learning: Feature Extraction
    7. Class Imbalance

In [ ]:
import h5py
import numpy as np
import math

import gc

import matplotlib.pyplot as plt
from matplotlib import cm, colormaps
from matplotlib.ticker import PercentFormatter

from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms.functional as TF
from torchinfo import summary

import segmentation_models_pytorch as smp

from functools import reduce

from IPython.display import IFrame

np.random.seed(3108)

## 3. Segmentation task

Load the data (and some helper functions) from the previous section.

In [ ]:
filename = '../data/dynamic_earth_data.h5'
dataset = h5py.File(filename, 'r')
planet_train = np.array(dataset['biromaru_planet'])
planet_test = np.array(dataset['guadeloupe_planet'])
labels_train = np.array(dataset['biromaru_labels'])
labels_test = np.array(dataset['guadeloupe_labels'])

def true_color(X):
    # B2 = blue, B3 = green, and B4 = red
    blue = ((X[:,:,0] - np.min(X[:,:,0])) / (np.max(X[:,:,0]) - np.min(X[:,:,0])))
    green = ((X[:,:,1] - np.min(X[:,:,1])) / (np.max(X[:,:,1]) - np.min(X[:,:,1])))
    red = ((X[:,:,2] - np.min(X[:,:,2])) / (np.max(X[:,:,2]) - np.min(X[:,:,2])))
    return np.dstack((red, green, blue))  

color_map = cm.get_cmap("Accent", lut=7)

labels=['impervious surface',
       'agriculture',
       'forest & other vegetation',
       'wetlands',
       'soil',
       'water',
       'snow & ice']

def labels2rgb(
    X,
    colors=color_map.colors,
    categorical=False):
    Z = np.zeros((X.shape[0], X.shape[1], 3))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            if categorical:
                Z[i,j] = colors[int(X[i,j]), :3]
            else:
                Z[i,j] = colors[np.argmax(X[i,j]), :3]
    return Z



In [ ]:
index = 0

fig=plt.figure(figsize=(12, 6))
fig.add_subplot(1, 2, 1)
plt.imshow(true_color(planet_train[index]))
plt.title('Planet Scope')
fig.add_subplot(1, 2, 2)
plt.imshow(labels2rgb(labels_train[index]), cmap=color_map)
plt.title('Labels')
cbar = plt.colorbar(ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)
plt.show()


fig=plt.figure(figsize=(12, 6))
fig.add_subplot(1, 2, 1)
plt.imshow(true_color(planet_test[index]))
plt.title('Planet Scope')
fig.add_subplot(1, 2, 2)
plt.imshow(labels2rgb(labels_test[index]), cmap=color_map)
plt.title('Labels')
cbar = plt.colorbar(ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)
plt.show()

In [ ]:
cls_train = np.argmax(labels_train, axis=-1)

cls_test = np.argmax(labels_test, axis=-1)

plt.hist([cls_train.flatten(), cls_test.flatten()], label=["Train", "Test"], density=True, bins=np.arange(0,8)-0.5)
plt.legend()
plt.show()

### 3.1. Random Forest

Will train a random forest classifier to predict on a pixel level.

In [ ]:
# convert the one-hot encoded segmentation mask into a categorial mask
labels_train_cat = np.argmax(labels_train, axis=-1)

labels_test_cat = np.argmax(labels_test, axis=-1)

To speed-up the training, we will select a single image from planet, and then train a random forest classifier on a pixel level.

In [ ]:
y = np.reshape(labels_train_cat[0], (reduce(lambda x, y: x*y, labels_train_cat[0].shape)))
print("Labels flattened shape", y.shape)

X = np.reshape(planet_train[0], (reduce(lambda x, y: x*y, planet_train[0].shape[:-1]),
                                              reduce(lambda x, y: x*y, planet_train[0].shape[-1:])))
print("Train flattened shape", X.shape)

In [ ]:
rfc = RandomForestClassifier(random_state=3108)
rfc.fit(X, y)

In [ ]:
Z = np.reshape(planet_test[0], (reduce(lambda x, y: x*y, planet_test[0].shape[:-1]),
                                              reduce(lambda x, y: x*y, planet_test[0].shape[-1:])))

yz = rfc.predict(Z)
yz_img = np.reshape(yz.copy(), (labels_test_cat[0].shape[0], labels_test_cat[0].shape[1]))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(yz_img, categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

On the left side, the ground truth is show, while on the right side the predicted segmentation is shown. Qualitatively, the results are quite OK. So, how about a quantitative appraisal?

There are different metrics to evaluate the performance of a semantic segmentation:
1. Pixel accuracy: Is the percent of pixels in your image that are classified correctly.
2. Intersection-Over-Union (IoU, Jaccard Index): The IoU is the area of overlap between the predicted segmentation and the ground truth divided by the area of union between the predicted segmentation and the ground truth.
3. Dice coefficient (F1 Score): Is two times the area of overlap divided by the total number of pixels in both images.

![segmentation-metrics.png](segmentation-metrics.png)

In [ ]:
print("Accuracy: %5.4f" % metrics.accuracy_score(labels_test_cat[0].flatten(), yz))
print("IoU: %5.4f" % metrics.jaccard_score(labels_test_cat[0].flatten(), yz, average="weighted"))
print("F1 Score: %5.4f" % metrics.f1_score(labels_test_cat[0].flatten(), yz, average="weighted"))

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat[0].flatten(), yz, labels=list(range(0,len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

### 3.2. Fully Convolutional Network (FCN)


Before we get started, we will chop the data into smaller patches. Otherwise, my laptop will struggle to compute.

In [ ]:
def slice_tensor_into_patches(images, targets, patch_size=256):
    """
    Slices a batch of images and targets into non-overlapping patches.
    
    Inputs:
      - images:  Tensor of shape [B, H, W, Ch]
      - targets: Tensor of shape [B, H, W, Cl]
    
    Outputs:
      - patch_images:  Tensor of shape [B*H/patch_size, patch_size, patch_size, Ch]
      - patch_targets: Tensor of shape [B*H/patch_size, patch_size, patch_size, Cl]
      
    """
    B, H, W, C_img = images.shape
    _, _, _, C_tgt = targets.shape

    # Calculate how many patches fit vertically and horizontally (1024 / 256 = 4)
    n_h = H // patch_size
    n_w = W // patch_size
    total_patches = B * n_h * n_w  # 24 * 4 * 4 = 384
    
    # 1. Break the spatial axes (H, W) into (num_patches, patch_size)
    # New shape: (24, 4, 256, 4, 256, 4)
    img_broken = images.reshape(B, n_h, patch_size, n_w, patch_size, C_img)
    tgt_broken = targets.reshape(B, n_h, patch_size, n_w, patch_size, C_tgt)
    
    # 2. Permute axes to group the patch identifiers together at the front
    # Desired order: (Batch, n_h, n_w, patch_size, patch_size, Channels)
    img_perm = img_broken.transpose(0, 1, 3, 2, 4, 5)
    tgt_perm = tgt_broken.transpose(0, 1, 3, 2, 4, 5)
    
    # 3. Collapse the first 3 tracking dimensions into a single continuous batch index
    patch_images = img_perm.reshape(total_patches, patch_size, patch_size, C_img)
    patch_targets = tgt_perm.reshape(total_patches, patch_size, patch_size, C_tgt)
    
    return patch_images, patch_targets


def patches_to_full_images(patch_masks, original_batch_size=24, image_size=1024, patch_size=256):
    """
    Reassembles patch predictions back into full-sized image maps.
    
    Parameters:
      - patch_masks: Array or Tensor of shape (384, 256, 256) 
                     containing integer class predictions.
      - original_batch_size: 24 (Number of full images in the original batch)
      - image_size: 1024 (Height/Width of the full image)
      - patch_size: 256 (Height/Width of each individual patch)
      
    Returns:
      - full_images: NumPy array of shape (24, 1024, 1024)
    """
    # 1. Convert to a standard NumPy array if it is a PyTorch Tensor
    if hasattr(patch_masks, "numpy"):
        patch_masks = patch_masks.detach().cpu().numpy()
        
    # Calculate how many patches fit along each axis (1024 / 256 = 4)
    n_h = image_size // patch_size
    n_w = image_size // patch_size
    
    # 2. Reshape the flat patch list back into structured grid dimensions
    # Shape changes from (384, 256, 256) -> (24, 4, 4, 256, 256)
    grid_structure = patch_masks.reshape(original_batch_size, n_h, n_w, patch_size, patch_size)
    
    # 3. Permute the dimensions to group corresponding rows and columns together
    # Target order: (Batch, n_h, patch_size, n_w, patch_size)
    permuted_grid = grid_structure.transpose(0, 1, 3, 2, 4)
    
    # 4. Collapse the inner tracking dimensions back into full resolution limits
    full_images = permuted_grid.reshape(original_batch_size, image_size, image_size)
    
    return full_images


In [ ]:
X_train_patches, y_train_patches = slice_tensor_into_patches(planet_train, labels_train, patch_size=256)
X_train_patches = torch.from_numpy(X_train_patches)
# torch expects images to be [batch, channel, height, width]
X_train_patches = X_train_patches.permute(0, 3, 1, 2).float()
train_dataset = TensorDataset(X_train_patches, torch.from_numpy(y_train_patches))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)

X_test_patches, y_test_patches = slice_tensor_into_patches(planet_test, labels_test, patch_size=256)
X_test_patches = torch.from_numpy(X_test_patches)
X_test_patches = X_test_patches.permute(0, 3, 1, 2).float()
test_dataset = TensorDataset(X_test_patches, torch.from_numpy(y_test_patches))
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

This is how each pair (image tile and labels) look like...

In [ ]:
images, gts = next(iter(train_loader))

print("Images batch shape", images.shape)
print("labels batch shape", gts.shape)

fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(gts[0]), cmap=color_map)
axs[0].set_title('Ground truth')

axs[1].imshow(true_color(images[0].permute(1,2,0).numpy()))
axs[1].set_title('Tile')

plt.show()

Now, let us define our fully convolutional network model

In [ ]:
class FCNModel(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.2):
        super(FCNModel, self).__init__()
        
        # Block 1: Input channels = 4, Output channels = 32
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=4, out_channels=32, kernel_size=3, stride=1, padding=1), # padding=1 matches "same" for kernel=3
            nn.Dropout2d(p=dropout_rate),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU()
        )
        
        # Block 2: Input channels = 32, Output channels = 64
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.Dropout2d(p=dropout_rate),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU()
        )
        
        # Block 3: Input channels = 64, Output channels = num_classes
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1, stride=1),
            nn.Dropout2d(p=dropout_rate),
            nn.BatchNorm2d(num_features=num_classes)
        )


    def forward(self, x):
        # Expects input shape: (batch_size, 4, 1024, 1024)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)        
        return x


In [ ]:
def train_loop(model, data_loader, 
               epochs=5, device="cpu", 
               criterion = nn.CrossEntropyLoss(),
               optimizer = None):    
    train_loss_history = []
    
    model.train()
    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=0.001)

    # scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
           
    for epoch in range(epochs):
        running_loss = 0.0
        for images, targets in data_loader:
            # Move data tensors onto the target compute hardware (CPU or GPU)
            images, targets = images.to(device), targets.to(device)

            # convert one-hot to class
            targets_idx = torch.argmax(targets, dim=-1)
            
            # Reset optimization gradients
            optimizer.zero_grad()
            
            outputs = model(images)
            
            loss = criterion(outputs, targets_idx)
            
            # Backpropagation & weight updating
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)

            del images, targets, outputs
            gc.collect() 

        # scheduler.step()
        
        epoch_loss = running_loss / len(data_loader.dataset)
        print(f"Epoch [{epoch+1}/5] Completed - Average Loss: {epoch_loss:.4f}")
        train_loss_history.append(epoch_loss)

    return model

In [ ]:
fcn_model = FCNModel(num_classes=7, dropout_rate=0.2)

summary(fcn_model, input_size=(1, 4, 256, 256))

In [ ]:
fcn_model = train_loop(fcn_model, train_loader, epochs=5)

In [ ]:
def evaluate_patches(model, dataloader, device="cpu", num_classes=7):
    # 1. Flag evaluation mode (freezes BatchNorm/Dropout configurations)
    model.eval()
    model.to(device)

    total_correct_pixels = 0
    total_pixels = 0
    
    # Initialize trackers for mIoU and Dice Score calculation
    intersection = torch.zeros(num_classes, device=device)
    union = torch.zeros(num_classes, device=device)
    target_sum = torch.zeros(num_classes, device=device)
    pred_sum = torch.zeros(num_classes, device=device)

    all_predicted_masks = []

    # 2. Deactivate autograd graph history tracking to protect CPU RAM
    with torch.no_grad():
        for inputs, targets in dataloader:
            # Collapse one-hot targets along depth: [B, 256, 256, 7] -> [B, 256, 256]
            targets_idx = torch.argmax(targets, dim=-1).to(device).long()

            # 3. Model Inference (outputs raw logits)
            outputs = model(inputs)      
            
            # 4. Extract final predicted class indices per pixel
            preds = torch.argmax(outputs, dim=1) # Shape: [B, 256, 256]

            # Cache predictions directly on CPU memory
            all_predicted_masks.append(preds.cpu())

            # 5. Accumulate overall pixel-wise evaluations
            total_correct_pixels += (preds == targets_idx).sum().item()
            total_pixels += targets_idx.numel()

            # 6. Calculate class-specific overlap metrics
            for cls in range(num_classes):
                pred_mask = (preds == cls)
                target_mask = (targets_idx == cls)

                intersection[cls] += (pred_mask & target_mask).sum().item()
                union[cls] += (pred_mask | target_mask).sum().item()
                
                pred_sum[cls] += pred_mask.sum().item()
                target_sum[cls] += target_mask.sum().item()

            # Explicit clean-up pointer drop per batch item to maintain low memory profile
            del inputs, targets, targets_idx, outputs, preds

    # --- Final Metric Matrix Calculations ---
    global_accuracy = total_correct_pixels / total_pixels
    eps = 1e-7
    
    # Mean Intersection over Union (mIoU)
    iou_per_class = intersection / (union + eps)
    mean_iou = torch.mean(iou_per_class).item()

    # Mean Dice Coefficient / F1-Score
    dice_per_class = (2.0 * intersection) / (pred_sum + target_sum + eps)
    mean_dice = torch.mean(dice_per_class).item()

    # Concatenate all generated array patches together
    saved_masks = torch.cat(all_predicted_masks, dim=0) # Shape: [Total_Patches, 256, 256]
    gc.collect()

    print("Accuracy: %5.4f" % global_accuracy)
    print("IoU: %5.4f" % mean_iou)
    print("F1 Score: %5.4f" % mean_dice)
    
    return {
        "accuracy": global_accuracy,
        "mIoU": mean_iou,
        "mDice": mean_dice,
        "predictions": saved_masks  
    }


In [ ]:
results = evaluate_patches(fcn_model, test_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (384, 256, 256)

# put predicted patches back into full 1024x1024 grids 
pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

#### Do it yourself

Would the results improve if we standarize the input?

In [ ]:
planet_scaler = StandardScaler()
planet_scaler.fit(planet_train.reshape(-1,4))

planet_train_std = planet_scaler.transform(planet_train.reshape(-1,4))
planet_train_std = planet_train_std.reshape(planet_train.shape)

planet_test_std = planet_scaler.transform(planet_test.reshape(-1,4))
planet_test_std = planet_test_std.reshape(planet_test.shape)

X_train_patches, y_train_patches = slice_tensor_into_patches(planet_train_std, labels_train, patch_size=256)
X_train_patches = torch.from_numpy(X_train_patches)
X_train_patches = X_train_patches.permute(0, 3, 1, 2).float()
train_dataset_std = TensorDataset(X_train_patches, torch.from_numpy(y_train_patches))
train_loader_std = DataLoader(train_dataset_std, batch_size=4, shuffle=True, num_workers=0)

X_test_patches, y_test_patches = slice_tensor_into_patches(planet_test_std, labels_test, patch_size=256)
X_test_patches = torch.from_numpy(X_test_patches)
X_test_patches = X_test_patches.permute(0, 3, 1, 2).float()
test_dataset_std = TensorDataset(X_test_patches, torch.from_numpy(y_test_patches))
test_loader_std = DataLoader(test_dataset_std, batch_size=4, shuffle=False, num_workers=0)

In [ ]:
# TODO train the model using the standarized data and evaluate it


### 3.3. U-Net

The U-Net is convolutional network architecture for fast and precise segmentation of images. It was originally invented, and first used for biomedical image segmentation. Its architecture can be broadly seen as an encoder network followed by a decoder network. Unlike classification, where the end result of the the deep network is the only important thing, semantic segmentation not only requires discrimination at pixel level but also a mechanism to project the discriminative features learnt at different stages of the encoder onto the pixel space.

The *encoder* is usually a pre-trained classification network (e.g., VGG or ResNet) where convolution blocks followed by maxpool downsampling are applied to encode the input image into multiple feature representation spaces.

The *decoder* semantically project the discriminative features (lower resolution) learnt by the encoder onto the pixel space (i.e., higher resolution) to get a dense classification (aka. segmentation).

The following image introduces the U-Net architecture (image taken from [*U-Net: Convolutional Networks for Biomedical Image Segmentation*](https://arxiv.org/abs/1505.04597v1))

<img src="u-net-architecture.png" width="800">

In [ ]:
class DoubleConv(nn.Module):
    """(Convolution -> BatchNorm -> ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=4, num_classes=7, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # --- Encoder (Downsampling) Path ---
        curr_in_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(curr_in_channels, feature))
            curr_in_channels = feature

        # --- Bottleneck ---
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2) # e.g., 512 -> 1024

        # --- Decoder (Upsampling) Path ---
        # We loop backwards through our features list
        for feature in reversed(features):
            # Upsample using Transposed Conv
            self.ups.append(
                nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2)
            )
            # Followed by a Double Conv block. 
            # Note: feature * 2 because we concatenate skip connections from the encoder!
            self.ups.append(DoubleConv(feature * 2, feature))

        # --- Final Output Layer ---
        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # 1. Run through the Encoder
        for down in self.downs:
            x = down(x)
            skip_connections.append(x) # Store for skip connections
            x = self.pool(x)

        # 2. Run through the Bottleneck
        x = self.bottleneck(x)
        
        # Reverse skip connections list to pop them easily matching the decoder
        skip_connections = skip_connections[::-1]

        # 3. Run through the Decoder (Step by 2: Up-conv, then Double-conv block)
        for idx in range(0, len(self.ups), 2):
            # Upsample
            x = self.ups[idx](x)
            
            # Fetch corresponding skip connection from encoder
            skip_connection = skip_connections[idx // 2]
            
            # Concatenate along the channel dimension (dim=1)
            concat_x = torch.cat((skip_connection, x), dim=1)
            
            # Double Conv block
            x = self.ups[idx + 1](concat_x)

        # 4. Final 1x1 Conv to map channels to classes
        return self.final_conv(x)



In [ ]:
unet_model = UNet(in_channels=4, num_classes=7)

summary(unet_model, input_size=(1,4,256,256))

# Uncomment and run if you want... but to save some time, you can just go to the next block of code
# unet_model = train_loop(unet_model, train_loader)

In [ ]:
# Run the following code if you want to save your model
# torch.save(unet_model.state_dict(), "../models/unet_example.pth")

In [ ]:
# Run this code to load a saved model
unet_model = UNet(in_channels=4, num_classes=7)
state_dict = torch.load("../models/unet_example.pth", map_location=torch.device('cpu'))
unet_model.load_state_dict(state_dict)

In [ ]:
results = evaluate_patches(unet_model, test_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (x, 256, 256)

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

### 3.4. Pre-trained Models

You may have notice that the U-Net architecture (as define above) has more than 31 million trainable parameters! Therefore, an immense amount of data is needed to train the network (if you want to get a decent performance).

Since we don't have much data, and we want to save time, let's try to reuse a pre-trained model.

What is a pre-trained model? In plain English, it is a model that was trained using a data set, and its weights have been saved. Usually, the data set used to train is general, e.g., imagenet.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images_np, labels_np, hs_mean, hs_std):
        self.images = images_np
        self.labels = labels_np
        self.hs_mean = hs_mean
        self.hs_std = hs_std

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]      
        label = self.labels[idx]
        
        rgb = img[:3, :, :]
        hs = img[3:, :, :]

        # Normalize the RGB channels using the params from imagenet
        rgb = TF.normalize(rgb.float(), mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        hs = TF.normalize(hs.float(), mean=[self.hs_mean], std=[self.hs_std])

        img = torch.cat([rgb, hs], dim=0)
        
        return img, label


X_train_patches, y_train_patches = slice_tensor_into_patches(planet_train, labels_train, patch_size=256)
hs_mean = np.mean(X_train_patches[:,:,:,3:])
hs_std = np.std(X_train_patches[:,:,:,3:])
X_train_patches = torch.from_numpy(X_train_patches)
X_train_patches = X_train_patches.permute(0, 3, 1, 2).float()
train_custom_dataset = CustomDataset(X_train_patches, torch.from_numpy(y_train_patches), hs_mean, hs_std)
train_custom_loader = DataLoader(train_custom_dataset, batch_size=4, shuffle=True)


X_test_patches, y_test_patches = slice_tensor_into_patches(planet_test, labels_test, patch_size=256)
X_test_patches = torch.from_numpy(X_test_patches)
X_test_patches = X_test_patches.permute(0, 3, 1, 2).float()
test_custom_dataset = CustomDataset(X_test_patches, torch.from_numpy(y_test_patches),  hs_mean, hs_std)
test_custom_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

Now, we will create a U-Net like model using an *EfficientNet-B2* (Tan and Le, ICML 2019)  model as encoder, pre-trained on *imagenet*.

Below, you can find a high-level description of the *EfficientNet-B2* architecture (generated using AI).

<img src="efficientnet-b2.jpg" width="800">

Then, we will use the model to *encode* the input, and the add a *decoder* to upsample the compressed representation.

<img src="unet-efficientnet-b2.jpg" width="800">

But with a *small* modification. The input (number of channels) and output of our dataset are different!

In our case, the library will:
- Create a brand new, empty convolutional layer containing the exact number of channels you requested. It then copies the available pretrained ImageNet weights into the first 3 channel slots. The remaining channel (i.e., the 4 th) will be randomly initialized.
- Create a decode architecture that matches the encoder, including a final head matching the labels. The decoder will be randomly initialized.

In [ ]:
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2", # "resnet34",
    encoder_weights="imagenet",
    in_channels=4,   # Explicitly expects 4 input channels
    classes=7,       # Explicitly outputs 7 classes
    activation=None  # Keep raw logits for CrossEntropyLoss
)

summary(pre_unet_model, input_size=(1,4,256,256))

Let us check the zero-shot performance of the model.

In [ ]:
results = evaluate_patches(pre_unet_model, test_custom_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (x, 256, 256)

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

#### Do it yourself

Remember that *half* of the model was randomly initialized. Bad and/or good luck may have played an important part here... Repeat the experiment, i.e., initialize the model again, and evaluate the performance.

### 3.5. Transfer Learning: Fine Tuning

You learnt to load a pre-trained model, and most likely, you notice that you can benefit out-of-the-box of these models. However, most of the times, the pre-trained models were trained on completely different problems. Therefore, it makes sense to adapt the behavior of our model to the problem, aka. transfer-learning.

In this context, we will *fine tune* the pre-trained model. That means, we will train the whole model, but with different learning rates.

In [ ]:
encoder_params = pre_unet_model.encoder.parameters()
decoder_params = list(pre_unet_model.decoder.parameters()) + list(pre_unet_model.segmentation_head.parameters())

optimizer = optim.AdamW([
    {"params": encoder_params, "lr": 1e-5}, # Lower LR for pretrained backbone
    {"params": decoder_params, "lr": 1e-4}  # Higher LR for custom layers
], weight_decay=1e-4)

# Uncomment to fine tune the model
# pre_unet_model = train_loop(pre_unet_model, train_custom_loader, optimizer = optimizer)  

In [ ]:
# Run the following code if you want to save your model
# torch.save(pre_unet_model.state_dict(), "../models/pre_unet_fine_tuned.pth")

In [ ]:
# Run this code to load a saved model
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2", # "resnet34",
    encoder_weights="imagenet",
    in_channels=4,   # Explicitly expects 4 input channels
    classes=7,       # Explicitly outputs 7 classes
    activation=None  # Keep raw logits for CrossEntropyLoss
)
state_dict = torch.load("../models/pre_unet_fine_tuned.pth", map_location=torch.device('cpu'))
pre_unet_model.load_state_dict(state_dict)

In [ ]:
results = evaluate_patches(pre_unet_model, test_custom_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (x, 256, 256)

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

The results are not particularly good... why?

Fine tuning typically requires a large amount of data (we are updating the whole model). Alternatives?

### 3.6. Transfer Learning: Feature Extraction

An alternative to fine tuning the model is the so call *feature extraction*. In this case, we will freeze the whole pretrained encoder backbone, and will train only the new decoder and segementation head.

This approach works quite well on small datasets.



In [ ]:
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    in_channels=4, 
    classes=7,   
    activation=None
)

# Freeze the encoder
for param in pre_unet_model.encoder.parameters():
    param.requires_grad = False

# The decoder and segmentation head will still calculate gradients
for param in pre_unet_model.decoder.parameters():
    assert param.requires_grad == True

# Only pass parameters that actually require gradients to save memory and compute
trainable_params = [p for p in pre_unet_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

# Uncomment to fine tune the model
# pre_unet_model = train_loop(pre_unet_model, train_custom_loader, optimizer = optimizer) 

In [ ]:
# Run the following code if you want to save your model
# torch.save(pre_unet_model.state_dict(), "../models/pre_unet_feat_ext.pth")

In [ ]:
# Run this code to load a saved model
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2", # "resnet34",
    encoder_weights="imagenet",
    in_channels=4,   # Explicitly expects 4 input channels
    classes=7,       # Explicitly outputs 7 classes
    activation=None  # Keep raw logits for CrossEntropyLoss
)
state_dict = torch.load("../models/pre_unet_feat_ext.pth", map_location=torch.device('cpu'))
pre_unet_model.load_state_dict(state_dict)

In [ ]:
results = evaluate_patches(pre_unet_model, test_custom_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (x, 256, 256)

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

#### Do it yourself

The model was pretrained on imagenet, an RGB dataset. However, our dataset has 4 bands. That means, that the model added an additional 4th channel slice with random weights to the first convolutional layer. When we freezed the encoder, that part in particular does not *learn*. In this case, we can unfreeze the first convolution layer.

In [ ]:
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    in_channels=4, 
    classes=7,   
    activation=None
)

# Freeze the encoder
for param in pre_unet_model.encoder.parameters():
    param.requires_grad = False

# Unfreeze just the first convolution layer so your 4th channel can train!
for param in pre_unet_model.encoder._conv_stem.parameters():
    param.requires_grad = True


Now, complete the code and train the network. Do you notice the difference?

#### Do it yourself

So far, we have sticked to the four bands. But, why not using only the RGB information? Try it yourself!

#### Do it yourself

There are multiple pre-trained segmentation models available on the pytorch library. You can check the details at https://smp.readthedocs.io/en/latest/models.html

Moreover, we can setup those models using different encoders. Check https://smp.readthedocs.io/en/latest/encoders.html for more information.

Now your task is to set up your own model, train and evaluate it.

### 3.7. Class Imbalance

A common issue when training machine learning models (and specially deep learning ones) is class imbalance. In our particular case, we already observed the problem.

There are several ways of addressing the problem, but one of the most effective ones is leveraging region-based loss functions. What does it mean? In plain English, we will modify the cost funtion to *add* more importance to under-represented pixels.

In [ ]:
# Multi-class Dice Loss (ignores background pixel volume)
dice_loss = smp.losses.DiceLoss(
    mode='multiclass', 
    classes=None # Calculates loss across all classes dynamically
)

# Multi-class Focal Loss (focuses on hard-to-classify pixels)
focal_loss = smp.losses.FocalLoss(
    mode='multiclass'
)

# Combined Loss Baseline
def combined_loss(y_pred, y_true):
    return dice_loss(y_pred, y_true) + focal_loss(y_pred, y_true)

In [ ]:
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    in_channels=4, 
    classes=7,   
    activation=None
)

# Freeze the encoder
for param in pre_unet_model.encoder.parameters():
    param.requires_grad = False

# Unfreeze just the first convolution layer so your 4th channel can train!
for param in pre_unet_model.encoder._conv_stem.parameters():
    param.requires_grad = True

# The decoder and segmentation head will still calculate gradients
for param in pre_unet_model.decoder.parameters():
    assert param.requires_grad == True

# Only pass parameters that actually require gradients to save memory and compute
trainable_params = [p for p in pre_unet_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

# Uncomment to fine tune the model
# pre_unet_model = train_loop(pre_unet_model, train_custom_loader, optimizer = optimizer, criterion = combined_loss) 

In [ ]:
# Run the following code if you want to save your model
# torch.save(pre_unet_model.state_dict(), "../models/pre_unet_cls_loss.pth")

In [ ]:
# Run this code to load a saved model
pre_unet_model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    in_channels=4, 
    classes=7,     
    activation=None
)
state_dict = torch.load("../models/pre_unet_cls_loss.pth", map_location=torch.device('cpu'))
pre_unet_model.load_state_dict(state_dict)

In [ ]:
results = evaluate_patches(pre_unet_model, test_custom_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  # Shape: (x, 256, 256)

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

There is an improvement, yes. However, the performance is still rather bad. How if we start from the U-Net we already trained from scratch?

In [ ]:
# Run this code to load a saved model
unet_model = UNet(in_channels=4, num_classes=7)
state_dict = torch.load("../models/unet_example.pth", map_location=torch.device('cpu'))
unet_model.load_state_dict(state_dict)


# Freeze the encoder
for param in unet_model.ups.parameters():
    param.requires_grad = False

# The decoder and segmentation head will still calculate gradients
for param in unet_model.downs.parameters():
    assert param.requires_grad == True

# Only pass parameters that actually require gradients to save memory and compute
trainable_params = [p for p in unet_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

# Uncomment to fine tune the model
# unet_model = train_loop(unet_model, train_custom_loader, optimizer = optimizer, criterion = combined_loss) 

In [ ]:
# Run the following code if you want to save your model
# torch.save(unet_model.state_dict(), "../models/unet_cls_loss.pth")

In [ ]:
unet_model = UNet(in_channels=4, num_classes=7)
state_dict = torch.load("../models/unet_cls_loss.pth", map_location=torch.device('cpu'))
unet_model.load_state_dict(state_dict)

In [ ]:
results = evaluate_patches(unet_model, test_custom_loader, device="cpu", num_classes=7)
predicted_patches = results["predictions"]  

pred_full_masks = patches_to_full_images(predicted_patches, original_batch_size=24)

In [ ]:
conf_matrix = metrics.confusion_matrix(labels_test_cat.flatten(), pred_full_masks.flatten(), labels=list(range(len(labels))))
metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, 
                               display_labels=labels).plot(xticks_rotation="vertical")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6), layout='constrained')

axs[0].imshow(labels2rgb(labels_test[0]), cmap=color_map)
axs[0].set_title('Ground truth')

im2 = axs[1].imshow(labels2rgb(pred_full_masks[0,:,:], categorical=True), cmap=color_map)
axs[1].set_title('Predicted')

cbar = fig.colorbar(im2, ax=axs, location='right', ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)

plt.show()

What happened!!!

Let us check again the data distribution, and try to draw some conclusions.

In [ ]:
cls_train = np.argmax(labels_train, axis=-1)

cls_test = np.argmax(labels_test, axis=-1)

plt.hist([cls_train.flatten(), cls_test.flatten()], label=["Train", "Test"], density=True, bins=np.arange(0,8)-0.5)
plt.legend()
plt.show()

#### Do it yourself

Try to improve the results we got so far.